# Módulo 12 · Semana 12, Integración de datos
Juan Esteban Camacho Castro, Juan Diego Serpa Ruiz

## Carga las tres fuentes y revisa su forma general

In [1]:
import pandas as pd

In [2]:
df_ventas = pd.read_csv("S11_PD_Camacho_Serpa_DatasetTransformado.csv")
df_bodega = pd.read_csv("bodega_categorias_novamarket.csv") 
df_crm = pd.read_csv("crm_clientes_novamarket.csv")

In [3]:
df_ventas.shape

(599000, 55)

In [4]:
df_bodega.shape

(7, 5)

In [5]:
df_crm.shape

(411840, 6)

## Inspecciona los nombres de columna y detecta los conflictos de nombre

In [6]:
print(df_ventas.columns.tolist())

['id_pedido', 'id_cliente', 'fecha_pedido', 'canal_compra', 'metodo_pago', 'ciudad_tienda', 'codigo_postal', 'categoria_producto', 'producto', 'precio_unitario', 'unidades_vendidas', 'monto_compra', 'edad_cliente', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad', 'comentario_cliente', 'peso_pedido_kg', 'fecha_actualizacion_stock', 'fuga_cliente', 'nivel_satisfaccion_imputado', 'ciudad_tienda_imputado', 'correo_cliente_imputado', 'comentario_cliente_imputado', 'z_monto_compra', 'es_venta_corporativa', 'es_precio_alto', 'cat_Belleza', 'cat_Deportes', 'cat_Electrónica', 'cat_Hogar', 'cat_Juguetería', 'cat_Moda', 'pago_Efectivo', 'pago_Paypal', 'pago_Tarjeta', 'pago_Transferencia', 'canal_App', 'canal_Marketplace', 'canal_Tienda', 'canal_Web', 'nivel_lealtad_cod', 'ciudad_tienda_cod', 'codigo_postal_agrupado', 'anio_pedido', 'mes_pedido', 'trimestre_pedido', 'dia_semana_pedido', 'es_fin_semana_pedido', 'longitud_comentario', 'num_palabras_comentario', 'tiene_comentario', 'menciona_

In [7]:
df_bodega.columns.tolist()

['categoria',
 'proveedor_principal',
 'bodega_asignada',
 'dias_reposicion',
 'stock_minimo']

In [8]:
df_crm.columns.tolist()

['email_cliente',
 'ciudad_residencia',
 'fecha_inscripcion_club',
 'puntos_fidelidad_actuales',
 'segmento_crm',
 'acepta_promociones']

### Conflictos de nombre detectados

Hay un conflicto entre la columna de df_ventas "categoria_producto" y la columna de df_bodega "categoria", aunque ambas representen a que categoría pertenece un producto, tal como juguete u otra
El segundo conflicto es entre df_ventas "correo_cliente" y "email_cliente", aunque ambas representen el correo electrónico del cliente, que también se le llama email (electronic mail)

## Verifica que las llaves de las tablas de referencia no estén repetidas

In [9]:
df_bodega["categoria"].duplicated().sum()

np.int64(0)

In [10]:
df_crm["email_cliente"].duplicated().sum()

np.int64(0)

## Estandariza las llaves de unión en las tres tablas

In [11]:
df_ventas["categoria_producto"] = df_ventas["categoria_producto"].str.strip().str.lower()
df_bodega["categoria"] = df_bodega["categoria"].str.strip().str.lower()

In [12]:
df_ventas["correo_cliente"] = df_ventas["correo_cliente"].str.strip().str.lower()
df_crm["email_cliente"] = df_crm["email_cliente"].str.strip().str.lower()

In [16]:
sorted(df_ventas["categoria_producto"].unique())

['belleza', 'deportes', 'electrónica', 'hogar', 'juguetería', 'moda']

In [17]:
sorted(df_bodega["categoria"].unique())

['deportes',
 'electrónica',
 'hogar',
 'juguetería',
 'mascotas',
 'moda',
 'papelería']

## Resuelve el conflicto de nombre y une ventas con bodega

In [19]:
df_bodega = df_bodega.rename(columns={"categoria": "categoria_producto"})

In [20]:
filas_antes = df_ventas.shape[0]

In [21]:
df_paso1 = pd.merge(df_ventas, df_bodega, on="categoria_producto", how="left")

In [27]:
print(filas_antes, df_paso1.shape[0])

599000 599000


## Resuelve el conflicto de nombre y une el resultado con el CRM

In [24]:
df_crm = df_crm.rename(columns={"email_cliente": "correo_cliente"})

In [25]:
df_integrado = pd.merge(df_paso1, df_crm, on="correo_cliente", how="left")

In [26]:
print(filas_antes, df_integrado.shape[0])

599000 599000


## Cuantifica y documenta los registros sin coincidencia

In [28]:
df_integrado["proveedor_principal"].isnull().sum()

np.int64(100218)

In [29]:
df_integrado["ciudad_residencia"].isnull().sum()

np.int64(201067)

### Registros sin coincidencia

Hay 100218 registros no encontraron categoría en bodega, posiblemente porque hay categorías nuevas como se ve en unas celdas más arriba, hay categorías nuevas como "mascotas" y "papelería" que están en bodega pero no en ventas, pero la categoría "belleza" no está en bodega pero si en ventas.

Mientras que hay 201067 registros de clientes que posiblemente no están en el club, pero igualmente tienen ventas, esto pudo haber generado la gran cantidad de nulos que se ven.

## Resuelve el conflicto de valor entre ciudad_tienda y ciudad_residencia

### Conflicto de valor: ciudad_tienda vs. ciudad_residencia

Porque no necesariamente donde compró un cliente, es donde vive según el CRM. Pudo haber comprado en otra ciudad como Cali, o inclusive otro país como USA por poner un ejemplo, pero sigue siendo una venta válida, mientras que su dirección de CRM dice que vive en Bogotá. No invalida para nada la venta, solo es una diferencia importante que hacer para no hacer un join entre ambas columnas que aunque representan una localización, sus contextos son diferentes.

## Exporta tu dataset integrado y guarda el notebook

In [30]:
df_integrado.to_csv("S12_PD_Apellido_DatasetIntegrado.csv", index=False)